# 03 — Classical model tuning

**Goal:** tune each of the six model families with Optuna and record the best hyperparameters in
`params.yaml: classical`, which `dvc repro` then trains with.

**Protocol:**
- Objective: mean **PR-AUC over 5-fold `StratifiedGroupKFold` on `group_id`** (train split only),
  the same folds as notebook 02. Grouped folds matter: with plain k-fold, Optuna would reward
  memorising scam templates, and the leak would come back.
- Deterministic: seeded TPE sampler, seeded models, fixed folds. Re-running gives the same studies.
  Studies are saved in `optuna.db`, so an interrupted run resumes where it stopped.
- Trials run one at a time (keeps TPE deterministic); the 5 folds of each trial run in parallel.
- The val split is not used here. It is kept for threshold tuning and model comparison in the
  pipeline. **The test split is not touched.**

This notebook and `uv run poe tune` call the same `fjd.models.tune.run_tuning`. Both write
`reports/tuning/<model>.json` and log one MLflow run per model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import yaml

from fjd.data.io import read_split
from fjd.models.classical import MODELS
from fjd.models.cv import cross_val_pr_auc, group_folds
from fjd.models.tune import run_tuning
from fjd.utils.config import load_params
from fjd.utils.plotting import SERIES, set_style
from fjd.utils.seed import set_seed
from fjd.utils.tracking import setup_mlflow

set_style()
optuna.logging.set_verbosity(optuna.logging.WARNING)
params = load_params()
SEED = params["seed"]
N_TRIALS = params["tuning"]["n_trials"]  # lower this for a quick look; the full run takes hours
set_seed(SEED)
print("MLflow:", setup_mlflow())

folds = group_folds(read_split("train"), params["features"], params["tuning"]["cv_folds"], SEED)

## 1. Tune every family

Slow: the linear models and Naive Bayes take minutes, the tree models take longer.

In [ ]:
studies, summaries = {}, {}
for name in MODELS:
    studies[name], summaries[name] = run_tuning(name, folds, N_TRIALS, SEED, "03_classical_tuning")
    s = summaries[name]
    print(f"{name:14s} CV PR-AUC {s['cv_pr_auc']:.3f} ({s['trials']} trials)")

## 2. Tuned vs default

The default-hyperparameter score uses the same folds as the studies.

In [ ]:
compare = pd.DataFrame(
    {
        name: {
            "default": np.mean(cross_val_pr_auc(name, {}, folds, SEED)),
            "tuned": summaries[name]["cv_pr_auc"],
            "tuned fold std": np.std(summaries[name]["cv_pr_auc_folds"]),
        }
        for name in MODELS
    }
).T
compare["gain"] = compare["tuned"] - compare["default"]
compare = compare.sort_values("tuned", ascending=False)
compare.style.format("{:.3f}").format("{:+.3f}", subset="gain")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharey=True)
for ax, name in zip(axes.flat, compare.index, strict=True):
    trials = studies[name].trials_dataframe(attrs=("number", "value", "state"))
    done = trials[trials["state"] == "COMPLETE"]
    ax.scatter(done["number"], done["value"], s=12, color=SERIES[0], alpha=0.6)
    ax.plot(done["number"], done["value"].cummax(), color=SERIES[1], lw=1.5)
    ax.axhline(compare.loc[name, "default"], color="grey", lw=1, ls="--")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("trial")
axes[0, 0].set_ylabel("CV PR-AUC")
axes[1, 0].set_ylabel("CV PR-AUC")
fig.suptitle(
    "Optuna studies (dots = completed trials, line = best so far, dashed = defaults)",
    x=0.01,
    ha="left",
    fontweight="bold",
)
plt.tight_layout()
plt.show()

## 3. Copy into `params.yaml`

Paste under `classical:`, then run `uv run dvc repro`. The pipeline retrains every changed model,
tunes its threshold on val and evaluates it.

In [ ]:
print(yaml.safe_dump({n: summaries[n]["best_params"] for n in MODELS}, sort_keys=False))

## 4. Takeaways

Numbers from the runs on 2026-10-20 to 10-23 (seed 42, 40 trials per model). Pipeline val/test
numbers are from `dvc repro` with the tuned `params.yaml`.

1. **Tuning closes the gap between families:** CV PR-AUC LinearSVM 0.558, LogReg 0.551,
   Random forest 0.549, LightGBM 0.547, XGBoost 0.541, Naive Bayes 0.497. The top five are within
   0.02, far below the fold-to-fold spread (std 0.12–0.20). No family wins on CV alone.
2. **Biggest gains:** Naive Bayes +0.20 (alpha 1 → 0.005: the default smoothing drowns the rare
   n-grams), Random forest +0.11 (≈2k χ² features instead of 5k, 800 trees), LogReg +0.06
   (C 1 → ≈100: much weaker regularisation lets the tabular/one-hot columns matter, see notebook 02).
   LinearSVM was already at its optimum (C ≈ 1.1); the boosters gained about 0.025.
3. **Trees and linear models fail differently:** on the four ordinary folds the tree models lead
   (0.58–0.69 vs 0.58–0.64), but on fold 3, the fold of unfamiliar fraud groups, they collapse to
   0.18–0.19 while the linear models keep 0.31. Trees memorise campaign-specific n-grams; the
   linear models generalise better to new campaigns. That matters for a deployed detector, and is
   worth weighing alongside the mean in notebook 05.
4. **Pipeline (val / test PR-AUC):** LinearSVM 0.701 / 0.794, LogReg 0.699 / 0.791,
   LightGBM 0.686 / 0.809, XGBoost 0.575 / 0.804, Random forest 0.604 / 0.801,
   Naive Bayes 0.525 / 0.694. At the F1-tuned val threshold, test precision is 0.91–0.93 for the
   linear and boosted models, with recall 0.61–0.68.
5. **Val and test disagree with CV:** both splits have ~102 fraud rows, so one or two fraud groups
   move PR-AUC by several points (test > val for every model). Model choice should rest on the CV
   mean and robustness (point 3), not on the test ranking; test stays a final report. XGBoost
   shows it: turning off its sampling moved CV PR-AUC by 0.002 but val by 0.11 (0.686 → 0.575).
6. **Reproducibility shapes XGBoost:** its row/column sampling draws differently on Windows and
   Linux, so it is fixed off (`fjd.models.classical`). Without column sampling, memory grows with
   features × 2^depth, so its search is capped at 5k χ² features and depth 8 (LightGBM: 10k).
7. **Cost (8 cores):** building the 5 featurized folds takes ~15 min; then the linear models,
   Naive Bayes, LightGBM (~12 min) and XGBoost (~30 min) tune in minutes to half an hour, with
   5 folds in parallel and 2 booster threads per fold. Trees are capped at 600.